In [1]:
!pip install pennylane --upgrade
from IPython.display import clear_output
from IPython.display import clear_output
clear_output(wait=False)

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    cohen_kappa_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc,
    confusion_matrix
)


In [4]:
!pip install xlrd

In [5]:
train_df = pd.read_csv(
    'train_fetus_75_spanish.xls',
    header=None,
    encoding='latin1'
)

print(train_df.head())
print(train_df.shape)

   0      1      2      3      4      5      6      7      8      9      ...  \
0      0      0      0      0      0      0      0      0      0      0  ...   
1      0      0      0      0      0      0      0      0      0      0  ...   
2      1      1      1      1      1      1      1      1      1      1  ...   
3     33     46     24     23     43     38     36     45     56     48  ...   
4      0      0      0      0     34     98     80     68     67     71  ...   

   50167  50168  50169  50170  50171  50172  50173  50174  50175  50176  
0    150    175    188    180    155    139    136    119     92      0  
1     15     10      8      8      7      8      6     12     20      0  
2      5      5      6      6      6      6      5      5      6      0  
3     13     10      7      6      4      3      1      1      1      0  
4      0      0      0      0      0      0      0      0      0      0  

[5 rows x 50177 columns]
(9195, 50177)


In [6]:
test_df = pd.read_csv(
    'test_75_spanish (2).xls',
    header=None,
    encoding='latin1'
)

print(test_df.head())


   0      1      2      3      4      5      6      7      8      9      ...  \
0      1      1      1      1      1      1      1      1      1      1  ...   
1      0      0      0      0      0      0      0      0      0      0  ...   
2      1      1      1      1      1      1      1      1      1      1  ...   
3      0      0      0      0      0      0      0      0      0      0  ...   
4      0      0      0      0      0      0      0      0      0      0  ...   

   50167  50168  50169  50170  50171  50172  50173  50174  50175  50176  
0    109     98     87     99    110    103     92     96    111      0  
1     22     27     37     39     37     38     36     36     40      0  
2      2      2      2      2      2      2      2      2      2      0  
3    118     90     86     89     91     92     87     89     85      0  
4     36     39     48     59     63     54     38     29     28      0  

[5 rows x 50177 columns]


In [7]:
train_df.shape, test_df.shape

((9195, 50177), (3062, 50177))

In [8]:
X_train = train_df.iloc[:, :-1].to_numpy()
y_train = train_df.iloc[:, -1].to_numpy()

X_test = test_df.iloc[:, :-1].to_numpy()
y_test = test_df.iloc[:, -1].to_numpy()

In [9]:
# Assuming y_resampled contains your class labels
unique, counts = np.unique(y_train, return_counts=True)
class_distribution = dict(zip(unique, counts))
print("Class distribution before augmentation:", class_distribution)

# Identify the minority class
minority_class_label = min(class_distribution, key=class_distribution.get)
print("Minority class label:", minority_class_label)

Class distribution before augmentation: {0: 533, 1: 1289, 2: 780, 3: 1219, 4: 3160, 5: 2214}
Minority class label: 0


In [10]:
X_train = X_train / 255.0
X_test = X_test / 255.0

In [11]:
X_train = np.nan_to_num(X_train)
X_test = np.nan_to_num(X_test)

In [12]:
x_train = X_train.reshape(9195,224,224)
x_test = X_test.reshape(3062,224,224)

In [13]:
x_train.shape,x_test.shape

((9195, 224, 224), (3062, 224, 224))

In [14]:
X_train2 = np.stack([x_train]*3, axis=-1)
X_test2 = np.stack([x_test]*3, axis=-1)
X_train2.shape, X_test2.shape,y_test,y_train   # Repeat grayscale images to create 3 channels

((9195, 224, 224, 3),
 (3062, 224, 224, 3),
 array([0, 0, 0, ..., 5, 5, 5]),
 array([0, 0, 0, ..., 5, 5, 5]))

In [15]:
X_train2.shape = (9195, 224, 224, 3)
X_test2.shape  = (3062, 224, 224, 3)

In [16]:
y_train.shape = (9195,)
y_test.shape  = (3062,)

In [ ]:
Proposed method with rayleigh and gaussian results 

In [ ]:
# =========================================================
# QRABNet (Swin-Tiny): training, evaluation and speckle-robustness pipeline
#
# Setup:
#   pip install timm thop pennylane --break-system-packages
#
# Expects these objects to already exist in the session (as in the original
# notebook): X_train2, X_test2 (images), y_train, y_test (labels).
# =========================================================
import os
import json
import pickle
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as T
import matplotlib.pyplot as plt
import pennylane as qml
import timm
from torch.utils.data import DataLoader, Dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report

# =========================================================
# CONFIGURATION (values match the manuscript's training table)
# =========================================================
SEED = 42
NAME = "swin_tiny"
TIMM_NAME = "swin_tiny_patch4_window7_224"
IMG_SIZE = 224

EPOCHS = 30                 # fixed budget; no early stopping, final-epoch weights are evaluated
BATCH_SIZE = 32
LR = 3e-4
WEIGHT_DECAY = 3e-4
LABEL_SMOOTHING = 0.05

MAX_SEVERITY = 0.3          # training: Rayleigh severity ~ U(0, 0.3), resampled every batch
W_CONS, W_NCE, W_NREG = 0.15, 0.3, 0.1   # consistency / contrastive / noise-regression weights
NCE_TEMPERATURE = 0.07

NOISE_LEVELS = [0.0, 0.05, 0.2, 0.5, 1.0]   # test-side sweep (Rayleigh lambda / Gaussian sigma)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

SAVE_DIR = "proposed model with metrics"
EVAL_ONLY_CKPT = None       # set to an existing .pth path to re-evaluate without retraining

# =========================================================
# REPRODUCIBILITY / DEVICE
# =========================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print("Device:", device, "| AMP enabled:", USE_AMP)

out_dir = os.path.join(SAVE_DIR, NAME)
os.makedirs(out_dir, exist_ok=True)

# =========================================================
# DATA
# =========================================================
def prepare_data(X):
    X = X.clone().float() if isinstance(X, torch.Tensor) else torch.tensor(X, dtype=torch.float32)
    if X.shape[-1] == 3:                 # NHWC -> NCHW
        X = X.permute(0, 3, 1, 2)
    if X.max() > 1:
        X = X / 255.0
    return X

X_train2 = prepare_data(X_train2)
X_test2 = prepare_data(X_test2)
y_train = y_train.clone().detach().long() if isinstance(y_train, torch.Tensor) else torch.tensor(y_train, dtype=torch.long)
y_test = y_test.clone().detach().long() if isinstance(y_test, torch.Tensor) else torch.tensor(y_test, dtype=torch.long)


class AugmentedDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X, self.y, self.transform = X, y, transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.transform is not None:
            x = self.transform(x)
        return x, self.y[idx]


def make_loaders(img_size):
    """Loaders return images in [0,1]. ImageNet normalization is applied later
    (normalize_batch) so speckle noise is added to genuine [0,1] intensities."""
    train_transform = T.Compose([
        T.RandomResizedCrop(img_size),
        T.RandomHorizontalFlip(),
        T.RandomRotation(15),
        T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    ])
    eval_transform = T.Resize((img_size, img_size))
    train_loader = DataLoader(AugmentedDataset(X_train2, y_train, train_transform),
                              batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    test_loader = DataLoader(AugmentedDataset(X_test2, y_test, eval_transform),
                             batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
    return train_loader, test_loader


def normalize_batch(x):
    mean = x.new_tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
    std = x.new_tensor(IMAGENET_STD).view(1, 3, 1, 1)
    return (x - mean) / std

# =========================================================
# SPECKLE NOISE (operates on [0,1] images)
#   Rayleigh : x * (1 + lambda * (R_unit - 1)), R_unit = R / mean(R)   [training + test sweep]
#   Gaussian : x * (1 + N(0, sigma^2))                                 [test sweep only]
# Both are clipped to [0,1].
# =========================================================
def add_speckle_noise(x, severity=0.2):
    noise = torch.from_numpy(np.random.rayleigh(scale=1.0, size=x.shape)).to(x.device).type_as(x)
    noise = noise / noise.mean()
    return torch.clamp(x * (1 + severity * (noise - 1)), 0.0, 1.0)


def add_speckle_noise_gaussian(x, sigma=0.2):
    noise = torch.randn(x.shape, device=x.device, dtype=x.dtype) * sigma
    return torch.clamp(x * (1.0 + noise), 0.0, 1.0)

# =========================================================
# QUANTUM CIRCUIT (used only by the Quantum Feature Gate)
# =========================================================
n_qubits = 8
dev = qml.device("default.qubit", wires=n_qubits)


@qml.qnode(dev, interface="torch")
def quantum_circuit_standard(inputs, weights):
    qml.templates.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.templates.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

# =========================================================
# QRABNet MODULES
# =========================================================
class FeatureRecalibration(nn.Module):
    """SE-style recalibration on the pooled 768-d vector (reduction 16 -> 48-d bottleneck)."""
    def __init__(self, dim, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(dim, dim // reduction), nn.ReLU(),
            nn.Linear(dim // reduction, dim), nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.fc(x)


class QuantumFeatureGate(nn.Module):
    """768 -> 8 (tanh) -> two circuits (depth 2 and 3, outputs averaged) -> 768 (sigmoid),
    multiplied by a classical sigmoid gate computed from the input."""
    def __init__(self, dim):
        super().__init__()
        self.fc_in = nn.Linear(dim, n_qubits)
        self.fc_out = nn.Linear(n_qubits, dim)
        self.scale = nn.Linear(dim, dim)

    def forward(self, x, gate_qlayers):
        q_in = torch.tanh(self.fc_in(x))
        q_encoded = torch.stack([torch.tanh(layer(q_in).float()) for layer in gate_qlayers], dim=0).mean(0)
        gate = torch.sigmoid(self.fc_out(q_encoded))
        alpha = torch.sigmoid(self.scale(x))
        return x * (alpha * gate), q_encoded


class NoiseEstimator(nn.Module):
    """Two-token (classical feature, quantum-gate encoding) multi-head self-attention ->
    scalar noise estimate alpha in [0,1] and an 8-d noise-bias vector."""
    def __init__(self, dim, attn_dim=64, n_heads=4):
        super().__init__()
        self.proj_x = nn.Linear(dim, attn_dim)
        self.proj_q = nn.Linear(n_qubits, attn_dim)
        self.attn = nn.MultiheadAttention(embed_dim=attn_dim, num_heads=n_heads, batch_first=True)
        self.norm = nn.LayerNorm(attn_dim)
        self.alpha_head = nn.Sequential(
            nn.Linear(attn_dim * 2, 128), nn.ReLU(),
            nn.Linear(128, 1), nn.Sigmoid(),
        )
        self.bias_head = nn.Sequential(nn.Linear(1, n_qubits), nn.Tanh())

    def forward(self, x, q_gate):
        tokens = torch.cat([self.proj_x(x).unsqueeze(1), self.proj_q(q_gate).unsqueeze(1)], dim=1)
        attn_out, _ = self.attn(tokens, tokens, tokens)
        attn_out = self.norm(attn_out + tokens)
        noise_alpha = self.alpha_head(attn_out.reshape(attn_out.size(0), -1))
        noise_bias = self.bias_head(noise_alpha) * torch.pi
        return noise_alpha, noise_bias


class BottleneckMLP(nn.Module):
    """Classical MLP sub-layer of the adaptive bottleneck: (8 + 8) -> hidden -> 8.
    hidden = depth * 8 * 3 (72 for depth 3, 96 for depth 4)."""
    def __init__(self, depth):
        super().__init__()
        hidden = depth * n_qubits * 3
        self.net = nn.Sequential(
            nn.Linear(n_qubits * 2, hidden), nn.Tanh(),
            nn.Linear(hidden, n_qubits), nn.Tanh(),
        )

    def forward(self, q_in, noise_bias):
        return self.net(torch.cat([q_in, noise_bias], dim=-1))


class AdaptiveBottleneck(nn.Module):
    """768 -> 8 -> (two classical MLP sub-layers, averaged) -> 768; residual scaled by alpha."""
    def __init__(self, dim):
        super().__init__()
        self.fc_in = nn.Linear(dim, n_qubits)
        self.fc_out = nn.Linear(n_qubits, dim)

    def forward(self, x, noise_alpha, noise_bias, bottleneck_layers):
        q_in = torch.tanh(self.fc_in(x))
        encoded = torch.stack([torch.tanh(layer(q_in, noise_bias)) for layer in bottleneck_layers], dim=0).mean(0)
        return x + noise_alpha * self.fc_out(encoded)


class QuantumGuidedFusion(nn.Module):
    """Softmax attention over [gated, bottleneck, raw] branches, conditioned on noise_alpha."""
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(dim * 3 + 1, dim), nn.ReLU(),
            nn.Linear(dim, 3), nn.Softmax(dim=-1),
        )
        self.proj = nn.Linear(dim, dim)

    def forward(self, f_gate, f_bottleneck, f_raw, noise_alpha):
        w = self.attn(torch.cat([f_gate, f_bottleneck, f_raw, noise_alpha], dim=-1))
        fused = w[:, 0:1] * f_gate + w[:, 1:2] * f_bottleneck + w[:, 2:3] * f_raw
        return self.proj(fused)


class ProjectionHead(nn.Module):
    def __init__(self, dim, proj_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.ReLU(),
            nn.Linear(dim, proj_dim),
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

# =========================================================
# LOSSES
# =========================================================
def noise_weighted_info_nce(z_clean, z_noisy, noise_alpha, temperature=NCE_TEMPERATURE):
    B = z_clean.size(0)
    z = torch.cat([z_clean, z_noisy], dim=0)
    sim = torch.mm(z, z.t()) / temperature
    sim.masked_fill_(torch.eye(2 * B, dtype=torch.bool, device=z.device), float("-inf"))
    labels = torch.cat([torch.arange(B, 2 * B, device=z.device), torch.arange(0, B, device=z.device)])
    per_sample = F.cross_entropy(sim, labels, reduction="none")
    alpha_flat = noise_alpha.squeeze(1)
    pair_weight = torch.cat([torch.ones_like(alpha_flat), 1.0 - alpha_flat], dim=0)
    return (per_sample * pair_weight).mean()


def noise_regression_loss(noise_alpha, true_severity):
    return F.mse_loss(noise_alpha, torch.full_like(noise_alpha, true_severity))

# =========================================================
# BACKBONE + FULL MODEL
# =========================================================
class TimmFeatureWrapper(nn.Module):
    # Wrapper kept so state_dict keys (encoder.backbone.*) match previously saved checkpoints.
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, x):
        return self.backbone(x)


def build_encoder_and_dim(timm_name):
    try:
        backbone = timm.create_model(timm_name, pretrained=True, num_classes=0,
                                     global_pool="avg", aux_logits=False)
    except TypeError:
        backbone = timm.create_model(timm_name, pretrained=True, num_classes=0, global_pool="avg")
    return TimmFeatureWrapper(backbone), backbone.num_features


class QRABNet(nn.Module):
    """
    Image -> Swin-Tiny -> Feature Recalibration (SE-style) -> Quantum Feature Gate
          -> Noise Estimator -> Adaptive Bottleneck -> Quantum-Guided Fusion -> Classifier
    The projection head is used only during training (contrastive loss).
    Only the Quantum Feature Gate contains quantum circuits (depth 2 and 3 -> 120 parameters).
    """
    def __init__(self, encoder, feat_dim, num_classes=6, proj_dim=128):
        super().__init__()
        self.encoder = encoder
        self.channel_attn = FeatureRecalibration(feat_dim)
        self.gate = QuantumFeatureGate(feat_dim)
        self.noise_est = NoiseEstimator(feat_dim)
        self.bottleneck = AdaptiveBottleneck(feat_dim)
        self.fusion = QuantumGuidedFusion(feat_dim)
        self.proj_head = ProjectionHead(feat_dim, proj_dim)
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )
        self.gate_qlayers = nn.ModuleList([
            qml.qnn.TorchLayer(quantum_circuit_standard, {"weights": (2, n_qubits, 3)}),
            qml.qnn.TorchLayer(quantum_circuit_standard, {"weights": (3, n_qubits, 3)}),
        ])
        # Classical MLP sub-layers (attribute name kept for checkpoint compatibility).
        self.bottleneck_qlayers = nn.ModuleList([BottleneckMLP(depth=3), BottleneckMLP(depth=4)])

    def forward(self, x, return_proj=False):
        x = self.encoder(x)
        x = self.channel_attn(x)
        f_gate, q_gate = self.gate(x, self.gate_qlayers)
        noise_alpha, noise_bias = self.noise_est(x, q_gate)
        f_bottleneck = self.bottleneck(x, noise_alpha, noise_bias, self.bottleneck_qlayers)
        fused = self.fusion(f_gate, f_bottleneck, x, noise_alpha)
        logits = self.classifier(fused)
        if return_proj:
            return logits, self.proj_head(fused), noise_alpha
        return logits

# =========================================================
# MODEL COMPLEXITY / EFFICIENCY
# GFLOPs from thop do not include the PennyLane circuits (8-d input per sample;
# their FLOP contribution is negligible next to the backbone).
# =========================================================
def compute_model_stats(model, img_size, ckpt_path, batch_size=16, n_warmup=5, n_runs=20):
    stats = {
        "total_params": sum(p.numel() for p in model.parameters()),
        "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
    }
    stats["model_size_mib"] = (os.path.getsize(ckpt_path) / 1024 ** 2 if os.path.exists(ckpt_path)
                               else stats["total_params"] * 4 / 1024 ** 2)

    model.eval()
    single = normalize_batch(torch.rand(1, 3, img_size, img_size, device=device))
    batch = normalize_batch(torch.rand(batch_size, 3, img_size, img_size, device=device))

    try:
        from thop import profile
        with torch.no_grad():
            macs, _ = profile(model, inputs=(single,), verbose=False)
        stats["gflops"] = 2 * macs / 1e9
    except Exception as e:
        print(f"[warn] GFLOPs profiling failed ({e})")
        stats["gflops"] = None

    def timed(inp):
        with torch.no_grad():
            for _ in range(n_warmup):
                model(inp)
            if device.type == "cuda":
                torch.cuda.synchronize()
                torch.cuda.reset_peak_memory_stats(device)
            times = []
            for _ in range(n_runs):
                if device.type == "cuda":
                    torch.cuda.synchronize()
                t0 = time.time()
                model(inp)
                if device.type == "cuda":
                    torch.cuda.synchronize()
                times.append(time.time() - t0)
        return sum(times) / len(times)

    t1 = timed(single)
    stats["latency_ms_bs1"] = t1 * 1000
    tb = timed(batch)
    stats["latency_ms_batch"] = tb * 1000
    stats["batch_size"] = batch_size
    stats["throughput_fps"] = batch_size / tb
    stats["peak_gpu_memory_mb"] = (torch.cuda.max_memory_allocated(device) / 1024 ** 2
                                   if device.type == "cuda" else None)
    return stats

# =========================================================
# TRAINING (fixed EPOCHS, cosine schedule, no early stopping / checkpoint selection)
# Per-epoch test accuracy is logged for the learning curves only; it is never used
# to select weights or to stop training.
# =========================================================
def train(model, train_loader, test_loader):
    class_weights = compute_class_weight("balanced", classes=np.unique(y_train.numpy()), y=y_train.numpy())
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32).to(device),
                                    label_smoothing=LABEL_SMOOTHING)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    history = {k: [] for k in ["train_acc", "test_acc", "severity",
                               "loss_cls", "loss_cons", "loss_nce", "loss_nreg", "loss_total"]}

    for epoch in range(EPOCHS):
        model.train()
        correct = total = 0
        sums = dict(cls=0.0, cons=0.0, nce=0.0, nreg=0.0, total=0.0, sev=0.0)
        n_batches = 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            severity = random.uniform(0.0, MAX_SEVERITY)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits_clean, z_clean, _ = model(normalize_batch(x), return_proj=True)
                logits_noisy, z_noisy, alpha_noisy = model(normalize_batch(add_speckle_noise(x, severity)),
                                                           return_proj=True)
                loss_cls = criterion(logits_clean, y)
                loss_cons = F.mse_loss(logits_clean, logits_noisy)
                loss_nce = noise_weighted_info_nce(z_clean, z_noisy, alpha_noisy)
                loss_nreg = noise_regression_loss(alpha_noisy, severity)
                loss = loss_cls + W_CONS * loss_cons + W_NCE * loss_nce + W_NREG * loss_nreg

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            correct += (logits_clean.argmax(1) == y).sum().item()
            total += y.size(0)
            for k, v in zip(["cls", "cons", "nce", "nreg", "total", "sev"],
                            [loss_cls, loss_cons, loss_nce, loss_nreg, loss, severity]):
                sums[k] += float(v)
            n_batches += 1

        model.eval()
        t_correct = t_total = 0
        with torch.no_grad(), torch.cuda.amp.autocast(enabled=USE_AMP):
            for x, y in test_loader:
                x, y = x.to(device), y.to(device)
                t_correct += (model(normalize_batch(x)).argmax(1) == y).sum().item()
                t_total += y.size(0)

        train_acc, test_acc = 100 * correct / total, 100 * t_correct / t_total
        history["train_acc"].append(train_acc)
        history["test_acc"].append(test_acc)
        history["severity"].append(sums["sev"] / n_batches)
        for k in ["cls", "cons", "nce", "nreg", "total"]:
            history[f"loss_{k}"].append(sums[k] / n_batches)

        scheduler.step()
        print(f"[{NAME}] Epoch {epoch + 1:02d}/{EPOCHS}: Train {train_acc:.2f}% | Test {test_acc:.2f}% | "
              f"mean severity {sums['sev'] / n_batches:.3f}")

    history["epochs_ran"] = EPOCHS
    history["final_epoch_test_acc"] = history["test_acc"][-1]
    return history

# =========================================================
# EVALUATION
# =========================================================
def regime(level):
    return "clinical" if level <= 0.2 else "hard" if level <= 0.5 else "extreme"


def robustness_sweep(model, test_loader, noise_fn, label):
    accs = []
    for level in NOISE_LEVELS:
        set_seed(SEED)   # identical noise draws for every run of the sweep
        correct = total = 0
        with torch.no_grad(), torch.cuda.amp.autocast(enabled=USE_AMP):
            for x, y in test_loader:
                x, y = x.to(device), y.to(device)
                preds = model(normalize_batch(noise_fn(x, level))).argmax(1)
                correct += (preds == y).sum().item()
                total += y.size(0)
        acc = 100 * correct / total
        accs.append(acc)
        print(f"[{NAME}] {label} {level:.2f} ({regime(level)}) -> Accuracy: {acc:.2f}%")
    return accs


def evaluate(model, test_loader, ckpt_path, history):
    model.eval()

    complexity = compute_model_stats(model, IMG_SIZE, ckpt_path)
    print(f"\n=== [{NAME}] Model complexity ===")
    for k, v in complexity.items():
        print(f"  {k}: {v}")

    preds_l, labels_l, probs_l = [], [], []
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=USE_AMP):
        for x, y in test_loader:
            probs = F.softmax(model(normalize_batch(x.to(device))).float(), dim=1)
            preds_l.append(probs.argmax(1).cpu())
            labels_l.append(y)
            probs_l.append(probs.cpu())
    y_pred = torch.cat(preds_l).numpy()
    y_true = torch.cat(labels_l).numpy()
    y_prob = torch.cat(probs_l).numpy()

    paths = {k: os.path.join(out_dir, f"{k}_{NAME}.npy") for k in ["y_true", "y_pred", "y_prob"]}
    np.save(paths["y_true"], y_true)
    np.save(paths["y_pred"], y_pred)
    np.save(paths["y_prob"], y_prob)

    report = classification_report(y_true, y_pred, digits=3)
    cm = confusion_matrix(y_true, y_pred)
    print(f"\n=== [{NAME}] Per-class performance (clean test set, final-epoch weights) ===")
    print(report)
    print("Confusion matrix:\n", cm)

    print(f"\n=== [{NAME}] Rayleigh sweep ===")
    rayleigh_acc = robustness_sweep(model, test_loader, add_speckle_noise, "Rayleigh")
    print(f"\n=== [{NAME}] Gaussian sweep (test-side only) ===")
    gaussian_acc = robustness_sweep(model, test_loader, add_speckle_noise_gaussian, "Gaussian")

    history.update({
        "classification_report": report,
        "confusion_matrix": cm.tolist(),
        "noise_levels": NOISE_LEVELS,
        "rayleigh_acc": rayleigh_acc,
        "gaussian_acc": gaussian_acc,
        "model_complexity": complexity,
        **{f"{k}_path": v for k, v in paths.items()},
    })
    return history

# =========================================================
# RUN
# =========================================================
if __name__ == "__main__" or True:   # notebook-friendly
    num_classes = len(torch.unique(y_train))
    ckpt_path = EVAL_ONLY_CKPT or os.path.join(out_dir, f"qrabnet_{NAME}_final.pth")

    train_loader, test_loader = make_loaders(IMG_SIZE)
    encoder, feat_dim = build_encoder_and_dim(TIMM_NAME)
    model = QRABNet(encoder, feat_dim, num_classes=num_classes).to(device)
    print(f"Feature dim: {feat_dim} | total params: {sum(p.numel() for p in model.parameters()):,} | "
          f"trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    if EVAL_ONLY_CKPT:
        # strict=False lets checkpoints from the earlier script (which also stored the unused
        # frozen shadow / probe modules) load into this cleaned model.
        missing, unexpected = model.load_state_dict(torch.load(EVAL_ONLY_CKPT, map_location=device), strict=False)
        print("Missing keys:", missing)
        print("Ignored keys (not part of the cleaned model):", len(unexpected))
        history = {"epochs_ran": None}
    else:
        history = train(model, train_loader, test_loader)
        torch.save(model.state_dict(), ckpt_path)   # final-epoch weights

    history = evaluate(model, test_loader, ckpt_path, history)

    with open(os.path.join(out_dir, f"history_{NAME}.pkl"), "wb") as f:
        pickle.dump(history, f)
    with open(os.path.join(out_dir, f"history_{NAME}.json"), "w") as f:
        json.dump(history, f, indent=2)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(NOISE_LEVELS, history["rayleigh_acc"], marker="o", label="Rayleigh (lambda)")
    axes[0].plot(NOISE_LEVELS, history["gaussian_acc"], marker="s", label="Gaussian (sigma)")
    axes[0].set_xlabel("Noise level")
    axes[0].set_ylabel("Test accuracy (%)")
    axes[0].set_title(f"Robustness to speckle noise ({NAME})")
    axes[0].legend()
    axes[0].grid(True)
    if history.get("train_acc"):
        axes[1].plot(range(1, len(history["train_acc"]) + 1), history["train_acc"], label="Train")
        axes[1].plot(range(1, len(history["test_acc"]) + 1), history["test_acc"], label="Test")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Accuracy (%)")
        axes[1].set_title("Learning curves")
        axes[1].legend()
        axes[1].grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"qrabnet_{NAME}_results.png"), dpi=150)
    plt.close(fig)

    print(f"\nHistory, arrays and plots saved in: {os.path.abspath(out_dir)}")
